# Agent-Based Epidemic Modeling
## Capstone Project Notebook

Agent-based models (ABMs) are computer simulations used to understand how individual behaviours and interactions give rise to outcomes at the population level. In epidemiology, ABMs are particularly useful for simulating how diseases spread through a population, based on the actions and characteristics of individuals.

In an ABM, each person is represented as an *agent*. Agents are assigned attributes such as age, vaccination status, health condition, or mobility. They follow rules that describe how they behave, interact with other agents, and respond to changes in their environment. Unlike compartmental models, which group individuals into categories (such as susceptible, infected, or recovered), ABMs simulate each individual separately. This allows for more detailed modelling of variation between people.

ABMs are *bottom-up* models: instead of starting with a population average, they build complex outcomes from simple individual behaviours. These behaviours and interactions can lead to **emergent effects** — patterns at the group level that are not obvious from the behaviour of individuals alone.

### ABMs vs. Differential Equation Models

| Feature | ODE (Compartmental) | Agent-Based |
|---------|---------------------|-------------|
| **Level of abstraction** | Aggregate numbers, homogeneous mixing | Individual agents, explicit interactions |
| **Heterogeneity** | All individuals in a compartment are identical | Each agent can have unique characteristics |
| **Contact structure** | Random (well-mixed) | Structured networks (families, schools, workplaces) |
| **Emergent behaviour** | Must be built into equations | Arises naturally from agent interactions |
| **Stochasticity** | Usually deterministic | Inherently stochastic — each run differs |
| **Computation** | Fast, analytically tractable | More complex, computationally intensive |
| **Data requirements** | Few average parameters | Detailed individual-level data |

One major strength of ABMs is their ability to simulate systems where individuals are not independent (e.g., household transmission), feedback loops exist (e.g., behaviour changes in response to risk), and hypothetical scenarios are of interest (e.g., interventions that cannot be tested in reality).

Despite their flexibility, ABMs have limitations: they can be computationally intensive, often require detailed data on individual behaviour, and because they can model scenarios that have not occurred, it can be difficult to validate results against real-world observations.

### In this notebook you will:
1. Build an **agent-based SIR model** from scratch, step by step
2. Implement **agent movement** on a 2D grid
3. Add **contact-based infection** and **recovery** rules
4. Visualise the spatial spread of an epidemic
5. Explore stochastic variability and parameter sensitivity
6. Lay the groundwork for required project extensions

---

## 0 · Setup

We import NumPy for array operations, Matplotlib for plotting, and Python's built-in `random` module for generating stochastic agent decisions. We define colour codes for the three SIR states and integer constants that will label each agent's health status throughout the simulation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import random

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

C_S = '#2ecc71'; C_I = '#e74c3c'; C_R = '#3498db'
STATE_S, STATE_I, STATE_R = 1, 2, 3

---
## 1 · Agent Movement

Each agent occupies a cell on a 2D grid and moves one step per time unit in a random cardinal direction (N, S, E, W). Agents cannot leave the grid boundaries — if a move would take them outside, they stay put.

This is a **random walk** — the simplest possible movement model. Despite its simplicity, it captures the key feature that agents explore their local neighbourhood over time, and the rate of spatial mixing depends on the grid size relative to the number of steps. On a small grid, agents mix quickly; on a large grid, spatial clusters persist longer.

The function below takes an agent's current position and returns a new position after one random step. We test it by running 1000 steps from a corner to verify no agent ever leaves the grid.

In [ ]:
def update_position(x, y, area, rng):
    """Move an agent one step in a random cardinal direction."""
    direction = rng.randint(1, 4)
    if direction == 1 and y < area: y += 1   # North
    elif direction == 2 and y > 0: y -= 1    # South
    elif direction == 3 and x > 0: x -= 1    # West
    elif direction == 4 and x < area: x += 1 # East
    return x, y

### Visualise a single agent's random walk

Before building the full epidemic model, it helps to see what a random walk looks like. Below we simulate 500 steps starting from the grid centre and plot the trajectory. The path wanders erratically but tends to stay within a region roughly proportional to $\sqrt{n}$ steps from the start — a fundamental property of random walks known as **diffusive scaling**.

In [ ]:
rng = random.Random(2)
x, y = 20, 20
trajectory = [(x, y)]
for _ in range(500):
    x, y = update_position(x, y, 40, rng)
    trajectory.append((x, y))

traj = np.array(trajectory)
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(traj[:, 0], traj[:, 1], lw=1, alpha=0.8, color='steelblue')
ax.plot(traj[0, 0], traj[0, 1], 'go', markersize=10, label='Start')
ax.plot(traj[-1, 0], traj[-1, 1], 'ro', markersize=10, label='End')
ax.set_xlim(0, 40); ax.set_ylim(0, 40)
ax.set_aspect('equal')
ax.set_title('Single Agent Random Walk (500 steps)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 2 · Infection Detection

In reality, disease transmission does not require two people to stand on exactly the same spot — proximity is what matters. A cough can travel several metres, and the risk of infection decreases with distance.

We model this with a **distance-dependent transmission probability**. Each infected agent within a radius $r_\text{infect}$ poses an independent transmission risk to a susceptible agent. The probability from a single infected neighbour at distance $d$ decays exponentially:

$$p_i = p_\text{infect} \cdot e^{-d\, /\, r_\text{infect}}$$

When multiple infected agents are nearby, each contributes an independent transmission attempt. The combined probability of getting infected by *at least one* of them is:

$$P = 1 - \prod_{i} (1 - p_i)$$

This captures two important real-world effects:
- **Distance matters** — close contacts are far more dangerous than distant ones
- **Multiple exposures compound** — being surrounded by several infected agents is much riskier than encountering just one

In [ ]:
def compute_infection_probability(person_idx, positions, states, p_infect, r_infect):
    """Compute infection probability based on proximity to infected agents.
    
    Each infected agent within radius r_infect contributes an independent
    transmission attempt with probability p_infect * exp(-dist / r_infect).
    Returns the combined probability: 1 - prod(1 - p_i).
    """
    if states[person_idx] != STATE_S:
        return 0.0
    
    dx = positions[:, 0].astype(float) - positions[person_idx, 0]
    dy = positions[:, 1].astype(float) - positions[person_idx, 1]
    distances = np.sqrt(dx**2 + dy**2)
    
    infected = states == STATE_I
    not_self = np.arange(len(states)) != person_idx
    in_range = distances <= r_infect
    
    mask = infected & not_self & in_range
    
    if not np.any(mask):
        return 0.0
    
    probs = p_infect * np.exp(-distances[mask] / r_infect)
    return 1.0 - np.prod(1.0 - probs)

---
## 3 · Health State Transitions

Each step, agents update their health state according to probabilistic rules:
- **Susceptible** agent with non-zero infection probability → **Infected** with that computed probability. The probability depends on how many infected agents are nearby and how close they are (see Section 2).
- **Infected** agent → **Recovered** with probability `p_recover` each step. The expected duration of illness is $1/p_\text{recover}$ steps.

The function `update_health` implements these transitions for a single agent. It now receives the pre-computed infection probability rather than a simple boolean. Recovered agents are permanently immune and never change state again.

In [ ]:
def update_health(state, infection_prob, p_recover, rng):
    """Compute the next health state for a single agent."""
    if state == STATE_S and infection_prob > 0:
        if rng.random() < infection_prob:
            return STATE_I
    elif state == STATE_I:
        if rng.random() < p_recover:
            return STATE_R
    return state

---
## 4 · Full Simulation

Now we assemble all components — movement, proximity-based infection, and health transitions — into a complete ABM simulation loop. At each time step the function:

1. Records the current counts of S, I, R agents and saves a snapshot of positions and states.
2. Updates health states: for each susceptible agent, computes the distance-dependent infection probability from all nearby infected agents within radius `r_infect`. Infected agents attempt recovery with probability `p_recover`.
3. Moves all agents: each agent takes one random step on the grid.

Health updates are computed into a separate `new_states` array so that all transitions within a single step are based on the same snapshot — no agent's state change affects another agent's transition in the same step.

The parameter `r_infect` controls the **infection radius** — the maximum distance at which transmission can occur. This, combined with `p_infect` (the base transmission probability at zero distance), determines how contagious the disease is. A larger radius means the disease can "jump" further, while a smaller radius confines spread to very close contacts.

The function returns a dictionary containing the epidemic curves (S, I, R counts over time) and full position/state histories for spatial visualisation.

In [ ]:
def simulate_abm(area=40, population=200, initially_infected=5,
                  steps=200, p_infect=0.25, p_recover=0.05,
                  r_infect=3.0, seed=42):
    """Run a full agent-based SIR simulation.
    
    Returns dict with keys: 'susceptible', 'infected', 'recovered',
    'positions', 'states', 'area'.
    """
    rng = random.Random(seed)
    positions = np.array([[rng.randint(0, area), rng.randint(0, area)]
                          for _ in range(population)], dtype=int)
    states = np.full(population, STATE_S, dtype=int)
    states[:initially_infected] = STATE_I
    
    s_hist, i_hist, r_hist = [], [], []
    pos_hist, st_hist = [], []
    
    for step in range(steps):
        s_hist.append(int(np.sum(states == STATE_S)))
        i_hist.append(int(np.sum(states == STATE_I)))
        r_hist.append(int(np.sum(states == STATE_R)))
        pos_hist.append(positions.copy())
        st_hist.append(states.copy())
        
        # Update health states
        new_states = states.copy()
        for p in range(population):
            prob = compute_infection_probability(p, positions, states, p_infect, r_infect)
            new_states[p] = update_health(states[p], prob, p_recover, rng)
        states = new_states
        
        # Move all agents
        for p in range(population):
            x, y = update_position(int(positions[p, 0]), int(positions[p, 1]), area, rng)
            positions[p] = [x, y]
    
    return {
        'susceptible': np.array(s_hist),
        'infected': np.array(i_hist),
        'recovered': np.array(r_hist),
        'positions': pos_hist,
        'states': st_hist,
        'area': area
    }

result = simulate_abm(seed=42)

### Population conservation check

A fundamental invariant of the SIR model is that the total population $S + I + R$ must remain constant at every time step — no agents are born, die (in this simple version), or leave the simulation. Below we verify this and display a summary of the simulation outcome.

In [ ]:
total = result['susceptible'] + result['infected'] + result['recovered']
population_conserved = np.all(total == 200)

print(f'Population conserved: {population_conserved}')
print(f'Peak infected: {np.max(result["infected"])} agents (step {np.argmax(result["infected"])})')
print(f'Total recovered by end: {result["recovered"][-1]}')

---
## 5 · Visualising the Epidemic

Two complementary views help us understand the ABM dynamics:

- **Epidemic curves** (left) — the time series of S, I, R agent counts. These are comparable to ODE-SIR output, but here they emerge from individual stochastic interactions rather than smooth differential equations. The curves are noisier and may differ between runs.
- **Spatial snapshot** (right) — the positions and states of all agents at the moment of peak infection. Unlike ODE models, the ABM shows *where* agents are: infected agents may cluster in certain regions while other areas remain unaffected. This spatial heterogeneity is a hallmark of agent-based models.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: epidemic curves
t = np.arange(len(result['susceptible']))
ax1.plot(t, result['susceptible'], color=C_S, lw=2, label='Susceptible')
ax1.plot(t, result['infected'], color=C_I, lw=2, label='Infected')
ax1.plot(t, result['recovered'], color=C_R, lw=2, label='Recovered')
ax1.set_xlabel('Time step')
ax1.set_ylabel('Number of agents')
ax1.set_title('Agent-Based SIR: Epidemic Curves', fontweight='bold')
ax1.legend()

# Right: spatial snapshot at peak infection
peak_step = np.argmax(result['infected'])
pos = result['positions'][peak_step]
st = result['states'][peak_step]
for state, color, label in [(STATE_S, C_S, 'S'), (STATE_I, C_I, 'I'), (STATE_R, C_R, 'R')]:
    mask = st == state
    ax2.scatter(pos[mask, 0], pos[mask, 1], c=color, s=15, alpha=0.7, label=label)
ax2.set_xlim(0, result['area']); ax2.set_ylim(0, result['area'])
ax2.set_aspect('equal')
ax2.set_title(f'Spatial Snapshot at Peak (step {peak_step})', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

### Spatial snapshots over time

The sequence below shows the grid at six time points. Each dot is an agent, coloured by state. In the early steps, infected agents (red) are clustered near their starting positions. As time progresses, the infection spreads outward through random encounters. By the later steps, most agents have recovered (blue), with only scattered susceptible agents (green) remaining — those who were lucky enough to avoid contact.

In [ ]:
snapshots = [0, 30, 60, 100]
fig, axes = plt.subplots(1, len(snapshots), figsize=(3.5 * len(snapshots), 3.5))

for ax, step in zip(axes, snapshots):
    pos = result['positions'][step]
    st = result['states'][step]
    for state, color in [(STATE_S, C_S), (STATE_I, C_I), (STATE_R, C_R)]:
        mask = st == state
        ax.scatter(pos[mask, 0], pos[mask, 1], c=color, s=8)
    nI = np.sum(st == STATE_I)
    ax.set_title(f't={step}, I={nI}', fontsize=10)
    ax.set_xlim(0, 40); ax.set_ylim(0, 40)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle('Spatial Spread Over Time', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

### Animated epidemic spread

The snapshots above show discrete moments, but an animation reveals the full spatial dynamics — how agents wander, how infection clusters grow and merge, and how the recovered barrier gradually forms. Below we run a fresh simulation and render each step as a frame, colouring agents by their SIR state.

In [ ]:
# --- Initialization ---
ANIM_STEPS = 200
anim_result = simulate_abm(area=40, population=200, initially_infected=5,
                            steps=ANIM_STEPS, p_infect=0.25, p_recover=0.05, seed=7)

fig, ax = plt.subplots(figsize=(6, 6))
ax.set_xlim(0, anim_result['area']); ax.set_ylim(0, anim_result['area'])
ax.set_aspect('equal')

# Initial empty scatter plots for each state
scat_s = ax.scatter([], [], c=C_S, s=15, alpha=0.7)
scat_i = ax.scatter([], [], c=C_I, s=15, alpha=0.7)
scat_r = ax.scatter([], [], c=C_R, s=15, alpha=0.7)

# --- Animation Logic ---

def update_plot(frame_number):
    """Update scatter plot with agent positions at this time step."""
    pos = anim_result['positions'][frame_number]
    st = anim_result['states'][frame_number]

    for state, scat in [(STATE_S, scat_s), (STATE_I, scat_i), (STATE_R, scat_r)]:
        mask = st == state
        scat.set_offsets(pos[mask] if np.any(mask) else np.empty((0, 2)))

    n_i = np.sum(st == STATE_I)
    ax.set_title(f'ABM Epidemic — Step {frame_number}/{ANIM_STEPS}  (I={n_i})')
    return [scat_s, scat_i, scat_r]

plt.close()

abm_animation = animation.FuncAnimation(
    fig, update_plot, frames=ANIM_STEPS, interval=60, blit=True
)

HTML(abm_animation.to_jshtml())

---
## 6 · Stochastic Variability

Different random seeds produce different epidemics. This is a key feature of ABMs — the same parameters can lead to very different outcomes due to the randomness in agent movement and infection events. Some runs may produce large outbreaks, while others may see the infection die out early if the initial infected agents happen to wander away from susceptible clusters.

Below we overlay 20 independent runs to visualise this variability. The spread of curves gives an intuitive sense of the **uncertainty** inherent in any single ABM prediction — and highlights why ensemble analysis (running many realisations) is essential when using stochastic models.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
peaks = []
for seed in range(1, 100):
    res = simulate_abm(seed=seed)
    ax.plot(res['infected'], color=C_I, lw=1, alpha=0.2)
    peaks.append(np.max(res['infected']))

ax.set_xlabel('Time step')
ax.set_ylabel('Infected agents')
ax.set_title('20 ABM Runs with Different Random Seeds', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7 · Parameter Exploration

How do the key parameters affect the epidemic? We explore three:

- **Infection probability** (`p_infect`) — the base transmission probability at zero distance. Higher values mean more aggressive disease spread, earlier and higher peaks, and a larger total number of infections.
- **Recovery probability** (`p_recover`) — controls how long agents remain infectious. Lower values mean longer illness duration, giving each infected agent more time to spread the disease before recovering.
- **Infection radius** (`r_infect`) — the maximum distance at which transmission can occur. A larger radius means each infected agent threatens more neighbours per step, dramatically accelerating spread. A very small radius approximates the original same-cell model.

The interplay of all three parameters — together with the spatial structure created by random movement — determines whether an outbreak takes off or fizzles out.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))

# Vary infection probability
for p_inf in [0.05, 0.15, 0.25, 0.50]:
    res = simulate_abm(p_infect=p_inf, seed=42)
    ax1.plot(res['infected'], lw=2, label=f'p_infect = {p_inf}')
ax1.set_xlabel('Time step'); ax1.set_ylabel('Infected')
ax1.set_title('Effect of Infection Probability', fontweight='bold')
ax1.legend(fontsize=9)

# Vary recovery probability
for p_rec in [0.01, 0.03, 0.05, 0.10]:
    res = simulate_abm(p_recover=p_rec, seed=42)
    ax2.plot(res['infected'], lw=2, label=f'p_recover = {p_rec}')
ax2.set_xlabel('Time step'); ax2.set_ylabel('Infected')
ax2.set_title('Effect of Recovery Probability', fontweight='bold')
ax2.legend(fontsize=9)

# Vary infection radius
for r_inf in [1.0, 2.0, 3.0, 5.0]:
    res = simulate_abm(r_infect=r_inf, seed=42)
    ax3.plot(res['infected'], lw=2, label=f'r_infect = {r_inf}')
ax3.set_xlabel('Time step'); ax3.set_ylabel('Infected')
ax3.set_title('Effect of Infection Radius', fontweight='bold')
ax3.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 8 · Your Tasks

You must implement the basic ABM SIR model (done above) and incorporate **at least one** of the following scenarios:

### Scenario A: Movement Patterns
Agents move freely but tend to converge at central locations (shopping centres, schools).
- Add "attraction points" on the grid that bias agent movement
- Agents near an attraction point have a probability of moving toward it
- Observe: do these hotspots create superspreading events?

### Scenario B: Intercommunity Spread
Agents live in separate communities connected by occasional travel.
- Create 2–4 separate grids, each with its own population
- A small fraction of agents "travel" between communities each step
- Observe: how does travel frequency affect the timing of outbreaks in different communities?

### Scenario C: Mask Compliance
Masks reduce infection probability, but not everyone complies.
- Assign each agent a boolean `wears_mask` attribute
- If a susceptible agent wears a mask, reduce `p_infect` by 50–70%
- Vary the compliance rate and measure the total infected

### Scenario D: Custom
Another feature of similar complexity (quarantine, testing, distance-based infection, etc.)

### Discussion points
- Show emergent behaviours (superspreading events, localised outbreaks)
- Analyse how different initial conditions affect disease spread
- Compare your simulation to real-world epidemic data
- Demonstrate results using a GIF or video

---
## Recommended Reading & Journal Club

### Foundational References

**1. Epstein, J. M. (2009)**
*Modelling to contain pandemics.*
Nature, 460, 687. [DOI](https://doi.org/10.1038/460687a)
→ Concise argument for why agent-based models are essential for pandemic preparedness.

**2. Bonabeau, E. (2002)**
*Agent-based modeling: Methods and techniques for simulating human systems.*
PNAS, 99(suppl 3), 7280–7287. [DOI](https://doi.org/10.1073/pnas.082080899)
→ Classic overview of ABM methodology.

---

### Journal Club Papers

**3. Kerr, C. C. et al. (2021)**
*Covasim: An agent-based model of COVID-19 dynamics and interventions.*
PLOS Computational Biology, 17(7), e1009149. [DOI](https://doi.org/10.1371/journal.pcbi.1009149)
→ Full-featured ABM used for real COVID-19 policy decisions. Good reference for realistic agent attributes.

**4. Perez, L. & Dragicevic, S. (2009)**
*An agent-based approach for modeling dynamics of contagious disease spread.*
International Journal of Health Geographics, 8, 50. [DOI](https://doi.org/10.1186/1476-072X-8-50)
→ Spatial ABM with GIS integration — bridges the gap between abstract and realistic models.

**5. Cuevas, E. (2020)**
*An agent-based model to evaluate the COVID-19 transmission risks in facilities.*
Computers in Biology and Medicine, 121, 103827. [DOI](https://doi.org/10.1016/j.compbiomed.2020.103827)
→ Facility-level ABM modelling indoor transmission — relevant to Scenario A (hotspots).

**6. Chang, S. L. et al. (2020)**
*Modelling transmission and control of the COVID-19 pandemic in Australia.*
Nature Communications, 11, 5710. [DOI](https://doi.org/10.1038/s41467-020-19393-6)
→ Large-scale ABM with mobility data and multiple intervention strategies.